# 共有トークナイザアーティファクトの Hub への切り出し(Promoting Canonical Tokenizer Artifacts)

## 目的

複数のトピック(005・006・008・009 以降)で再利用するトークナイザを、それぞれ独立した
Hugging Face Hub リポジトリへ切り出す。現在、英語バイトレベル BPE(byte-level Byte Pair
Encoding)は 008 のモデルリポジトリ(`kojikojiprg/ai-theories-small-gpt-en`)に同梱される
形で配置されているが、009 はモデルを必要とせずトークナイザのみを必要とするため、この
依存関係は不自然である。このパターンは今後 010 以降でも繰り返しうるため、言語・ドメインごとに
専用リポジトリへ切り出す。

対象とする 3 つのアーティファクトの由来・位置づけは以下の通り。

| 言語 / ドメイン | 種類 | 設定 | 由来 | 位置づけ |
|---|---|---|---|---|
| 英語(en) | バイトレベル BPE | 語彙サイズ 8192 | 008 のモデルリポジトリに同梱済みのものをそのまま再利用(再学習しない) | 006 の本番実験で選定された条件 |
| 日本語(ja) | 文字レベル(Character-level) | ― | 006 の本番実験で日本語の勝者となった条件。006 が実際に使ったコーパスから、006 と完全に同一の手順で再構築する | 006 の本番実験で選定された条件 |
| コード(code) | バイトレベル BPE | 語彙サイズ 8192、`max_chunk_bytes=64`(英語と同一設定) | `load_code_corpus()`(リポジトリ自身の`src/`を連結するコーパス) | 暫定(provisional)。006 はコードドメインで言語モデルの学習を行っておらず「正解」が存在しないため、英語と同じ設定を仮採用する |

BPE の学習・文字レベル語彙の構築はいずれも GPU を必要としない処理であるため、本番スケールの
まま(記事数・コーパス量を縮小せずに)ローカルで構築・検証する。Hugging Face Hub への実際の
アップロード呼び出しのみ、`HF_TOKEN`の認証を要する操作としてガードし(`DRY_RUN`・`IN_COLAB`)、
Google Colab で行う。

**対象外**: 006 は 5 つのトークナイザ条件を比較すること自体が実験の内容であるため変更しない。
009 のトークナイザ取得元の切り替えは、009 の Colab 本番実行(スケーリング則の学習グリッド)
完了後に別途行う(本ノートブックでは行わない)。

In [1]:
# 環境セットアップ(Google Colab)
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/kojikojiprg/ai-theories.git
    %cd ai-theories
    !pip install uv -q
    !uv pip install --system -r requirements.txt
# ローカル(Jupyter)実行時は、リポジトリルートで起動していればそのまま動く。

Cloning into 'ai-theories'...
remote: Enumerating objects: 538, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 538 (delta 12), reused 19 (delta 6), pack-reused 502 (from 1)
Receiving objects: 100% (538/538), 6.51 MiB | 20.37 MiB/s, done.
Resolving deltas: 100% (268/268), done.
/content/ai-theories
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 45.7 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 52 packages in 396ms
Prepared 28 packages in 58.42s
Uninstalled 10 packages in 1.00s
Installed 28 packages in 393ms
 - click==8.5.0
 + click==8.4.2
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.6.0
 + cuda-toolkit==13.0.3.0
 - filelock==3.32.4
 + filelock==3.32.2
 - fsspec==2025.3.0
 + fsspec==2026.7.0
 - matplotlib==3.10.0
 + matplotlib==3.11.1
 - numpy==2.1.3
 + numpy==2.5.2
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + 

In [2]:
import shutil
import subprocess
from pathlib import Path

from huggingface_hub import hf_hub_download

from src.data.text import (
    CharacterLevelTokenizer,
    load_code_corpus,
    load_wikipedia_corpus,
    save_character_level_tokenizer_json,
)
from src.data.tokenizer import (
    BPEIDTokenizer,
    learn_bpe,
    load_bpe_id_tokenizer_json,
    save_bpe_id_tokenizer_json,
    upload_tokenizer_artifact_to_hub,
)

DRY_RUN = False  # Claude Code はこの True 側のみ実行する(Colab で DRY_RUN=False に切り替えるとアップロードが実行される)

ROOT = Path(".")
CACHE_DIR = ROOT / ".cache" / "promote_canonical_tokenizers"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

## 英語(en)アーティファクトの再パッケージ

**再学習は行わない。** 008 のモデルリポジトリ(`kojikojiprg/ai-theories-small-gpt-en`)から
`tokenizer.json` を取得し、その内容をそのまま(バイト単位で)新しいリポジトリ用に使う。

In [3]:
EN_SOURCE_REPO_ID = "kojikojiprg/ai-theories-small-gpt-en"  # 008 のモデルリポジトリ
EN_TARGET_REPO_ID = "kojikojiprg/ai-theories-tokenizer-en"

en_tokenizer_json_path = Path(hf_hub_download(repo_id=EN_SOURCE_REPO_ID, filename="tokenizer.json"))

# パース可能であること・語彙サイズが期待通りであることを検証する(内容そのものの
# シリアライズ形式は変えないよう、アップロード用ファイルはこのオブジェクトから
# 再生成せず、ダウンロードしたファイルをそのままコピーする、下記参照)。
en_tokenizer = load_bpe_id_tokenizer_json(en_tokenizer_json_path)
print(f"英語トークナイザ取得元: {EN_SOURCE_REPO_ID}")
print(f"vocab_size = {en_tokenizer.vocab_size}")
assert en_tokenizer.vocab_size == 8192, "英語トークナイザの語彙サイズが 8192 と一致しない"

# ラウンドトリップ検証(短いサンプルテキストで確認する。取得したトークナイザを
# 再学習するわけではないため、008 の学習コーパス全体を再取得する必要はない)。
_en_roundtrip_sample = "The quick brown fox jumps over the lazy dog. Hello, world! 12345."
assert en_tokenizer.decode(en_tokenizer.encode(_en_roundtrip_sample)) == _en_roundtrip_sample
print("[OK] 英語トークナイザのラウンドトリップ検証")

# 009(scaling laws)は 2026-09-06 時点で Colab で本番実行中であり、009 自身が
# アップロードした tokenizer.json はまだ存在しない(009 はそもそも本番実行が完了しても
# 独自にアップロードするわけではなく、EN_SOURCE_REPO_ID から取得するだけである)。
# したがって「009 と tokenizer.json のバイト列が完全一致するか」の検証はスキップする
# (009 が取得する先も同じ EN_SOURCE_REPO_ID であるため、内容が異なる余地はそもそもない)。
print("[スキップ] 009 との tokenizer.json バイト列比較(009 は本ノートブックと同じ取得元を使うため)")

en_artifact_dir = CACHE_DIR / "en"
en_artifact_dir.mkdir(parents=True, exist_ok=True)
en_artifact_path = en_artifact_dir / "tokenizer.json"
shutil.copy(en_tokenizer_json_path, en_artifact_path)
print(f"アップロード用ファイルを書き出した: {en_artifact_path}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


tokenizer.json:   0%|          | 0.00/661k [00:00<?, ?B/s]

英語トークナイザ取得元: kojikojiprg/ai-theories-small-gpt-en
vocab_size = 8192
[OK] 英語トークナイザのラウンドトリップ検証
[スキップ] 009 との tokenizer.json バイト列比較(009 は本ノートブックと同じ取得元を使うため)
アップロード用ファイルを書き出した: .cache/promote_canonical_tokenizers/en/tokenizer.json


## 日本語(ja)アーティファクトの構築

**006 のノートブックを実際に読んで確認した構築手順**(推測ではない): 006 の 5.2 節・5.4 節
(`build_tokenizers()`)は、`CharacterLevelTokenizer` を `full_corpus[lang]`
(訓練・検証に分割する **前** のコーパス全体)から構築している。`VALIDATION_RATIO`
による訓練・検証分割は言語モデルの学習・評価にのみ使われ、文字レベル語彙の構築には
使われていない(検証テキストのみに出現する文字で符号化が失敗しないよう、コーパス全体を
使う設計、006 のコード内コメントを参照)。

また 006 が実際に呼んでいるのは `load_wikipedia_corpus("ja", CACHE_DIR)`
(既定マニフェスト `ja_006_pretraining.json`)であり、`load_japanese_corpus()`
(005 時点の別マニフェスト `ja_005_tokenizer_legacy.json` を使う薄いラッパー)ではない。
以下はこの実際の手順を再現する。

In [4]:
JA_TARGET_REPO_ID = "kojikojiprg/ai-theories-tokenizer-ja"

# 006 と同じキャッシュディレクトリ(.cache/006_corpus)を指定し、既に取得済みの記事
# キャッシュを再利用する(記事タイトル・リビジョン ID を固定しているため、再取得しても
# 内容は変わらない)。
JA_CACHE_DIR = ROOT / ".cache" / "006_corpus"
corpus_ja = load_wikipedia_corpus("ja", JA_CACHE_DIR)
print(f"corpus_ja: {len(corpus_ja):,} 文字 / {len(corpus_ja.encode('utf-8')):,} バイト")

# 006 の本番実行(LM_CORPUS_BYTES = 10**12、実質無制限)では、load_corpus_prefix() による
# バイト数での切り詰めは no-op である(実際のコーパスサイズがこれを大きく下回るため)。
# したがって corpus_ja は 006 の full_corpus["ja"] と完全に一致する。

ja_tokenizer = CharacterLevelTokenizer(corpus_ja)
print(f"vocab_size = {ja_tokenizer.vocab_size}")

# 006 の本番実行結果(セル出力: `ja/character: vocab_size=4654`)と一致することを確認する。
_EXPECTED_JA_VOCAB_SIZE = 4654
assert ja_tokenizer.vocab_size == _EXPECTED_JA_VOCAB_SIZE, (
    f"006 の本番実行結果({_EXPECTED_JA_VOCAB_SIZE})と一致しない: {ja_tokenizer.vocab_size}"
)
print(f"[OK] 006 の本番実行結果(vocab_size={_EXPECTED_JA_VOCAB_SIZE})と一致した")

# ラウンドトリップ検証(コーパス全体)。
assert ja_tokenizer.decode(ja_tokenizer.encode(corpus_ja)) == corpus_ja
print("[OK] 日本語トークナイザのラウンドトリップ検証(コーパス全体)")

ja_artifact_dir = CACHE_DIR / "ja"
ja_artifact_dir.mkdir(parents=True, exist_ok=True)
save_character_level_tokenizer_json(ja_tokenizer, ja_artifact_dir / "tokenizer.json")
print(f"アップロード用ファイルを書き出した: {ja_artifact_dir / 'tokenizer.json'}")

corpus_ja: 8,955,329 文字 / 24,575,245 バイト
vocab_size = 4654
[OK] 006 の本番実行結果(vocab_size=4654)と一致した
[OK] 日本語トークナイザのラウンドトリップ検証(コーパス全体)
アップロード用ファイルを書き出した: .cache/promote_canonical_tokenizers/ja/tokenizer.json


## コード(code)アーティファクトの構築

**暫定(provisional)扱い。** 006 はコードドメインで言語モデルの学習・トークナイザ条件の
比較を行っておらず、コードドメインにおける「正解」となる設定はまだ存在しない。ここでは
英語と同じ設定(バイトレベル BPE、語彙サイズ 8192、`max_chunk_bytes=64`)を仮採用する。

`load_code_corpus()` はリポジトリ自身の `src/` を実行時点の内容で連結するため、取得結果は
取得時点のリポジトリ内容に依存する(`load_code_corpus()` の docstring、および
コーディング規約の既存の注意書きと同じ理由による)。取得に使ったコミットハッシュを記録する。

In [5]:
CODE_TARGET_REPO_ID = "kojikojiprg/ai-theories-tokenizer-code"

code_corpus = load_code_corpus(".")
print(f"code_corpus: {len(code_corpus):,} 文字 / {len(code_corpus.encode('utf-8')):,} バイト")

code_commit_hash = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
# load_code_corpus() が実際に読むのは src/ 配下のみなので、コミットハッシュとの
# 対応が崩れているかどうかは src/ の変更有無だけで判定すれば十分(リポジトリ全体の
# 変更有無で判定すると、src/ に無関係な変更で誤って「dirty」と報告してしまう)。
_git_status = subprocess.run(
    ["git", "status", "--porcelain", "--", "src"], capture_output=True, text=True, check=True
).stdout
code_is_dirty = _git_status.strip() != ""
print(f"commit hash: {code_commit_hash}")
print(f"src/ に未コミットの変更があるか: {code_is_dirty}")

code_bpe = learn_bpe(code_corpus, vocab_size=8192, byte_level=True, max_chunk_bytes=64)
code_symbols = sorted(code_bpe.vocab)
code_symbol_to_id = {s: i for i, s in enumerate(code_symbols)}
code_tokenizer = BPEIDTokenizer(code_bpe, code_symbol_to_id)
print(f"vocab_size = {code_tokenizer.vocab_size}")

assert code_tokenizer.decode(code_tokenizer.encode(code_corpus)) == code_corpus
print("[OK] コードトークナイザのラウンドトリップ検証(コーパス全体)")

code_artifact_dir = CACHE_DIR / "code"
code_artifact_dir.mkdir(parents=True, exist_ok=True)
save_bpe_id_tokenizer_json(code_tokenizer, code_artifact_dir / "tokenizer.json")
print(f"アップロード用ファイルを書き出した: {code_artifact_dir / 'tokenizer.json'}")

code_corpus: 204,741 文字 / 271,810 バイト
commit hash: 05b684285b77ffd1a6b6f6fcb762cdd8e332b0d5
src/ に未コミットの変更があるか: False
vocab_size = 8192
[OK] コードトークナイザのラウンドトリップ検証(コーパス全体)
アップロード用ファイルを書き出した: .cache/promote_canonical_tokenizers/code/tokenizer.json


## モデルカードの作成

3 つのアーティファクトそれぞれについて、日本語メイン・英語併記のモデルカードを作成する。
由来トピックへのリンク、スクラッチ実装であること・研究教育目的であり品質保証がないこと、
コードアーティファクトについては暫定扱いである旨と理由・コーパスのコミットハッシュを含める。
ライセンスは 3 つとも `cc-by-nc-4.0` を採用する(008 のモデルカードの `mit` 表記の
修正は本ノートブックの対象外)。

In [6]:
_dirty_note = (
    "(取得時点で作業ツリーに未コミットの変更があったため、この commit hash が示す内容と\n"
    "厳密には一致しない可能性がある)"
    if code_is_dirty
    else ""
)

en_model_card = """---
language: en
license: cc-by-nc-4.0
tags:
- ai-theories
- tokenizer
- byte-level-bpe
---

# ai-theories 標準トークナイザ(英語・バイトレベル BPE)

`ai-theories`(https://github.com/kojikojiprg/ai-theories)プロジェクトの成果物。
バイトレベル Byte Pair Encoding(BPE、語彙サイズ 8192)のスクラッチ実装によるトークナイザ。

研究・教育目的で構築したものであり、品質保証は行っていない。商用・実運用での利用は
想定しない。

## 由来

[006. 小型 GPT の事前学習](https://github.com/kojikojiprg/ai-theories/blob/main/theories/02_pretraining/006_pretraining_small_gpt.ipynb)
の本番実験で、英語について 5 つのトークナイザ条件(文字レベル・バイトレベル BPE ×
4 語彙サイズ・Unigram 言語モデル)の中から選定された条件(バイトレベル BPE、語彙サイズ
8192)である。学習済みの語彙・マージ規則自体は
[008. デコーディング戦略](https://github.com/kojikojiprg/ai-theories/blob/main/theories/02_pretraining/008_decoding_strategies.ipynb)
の学習済みモデルに同梱されていたものをそのまま引き継いでおり、再学習は行っていない。

## 構成

- 方式: バイトレベル BPE(byte-level Byte Pair Encoding)
- 語彙サイズ: 8192
- 事前分割チャンクの最大バイト数(`max_chunk_bytes`): 64

`tokenizer.json` は `merges`・`vocab`・`byte_level`・`chunk_split_mode`・`max_chunk_bytes`・
`symbol_to_id` を含む(`src/data/tokenizer.py` の `load_bpe_id_tokenizer_json()` で
読み込める)。
"""

ja_model_card = f"""---
language: ja
license: cc-by-nc-4.0
tags:
- ai-theories
- tokenizer
- character-level
---

# ai-theories 標準トークナイザ(日本語・文字レベル)

`ai-theories`(https://github.com/kojikojiprg/ai-theories)プロジェクトの成果物。
文字単位(character-level)のトークナイザのスクラッチ実装。

研究・教育目的で構築したものであり、品質保証は行っていない。商用・実運用での利用は
想定しない。

## 由来

[006. 小型 GPT の事前学習](https://github.com/kojikojiprg/ai-theories/blob/main/theories/02_pretraining/006_pretraining_small_gpt.ipynb)
の本番実験で、日本語について 5 つのトークナイザ条件の中から選定された条件(文字レベル)
である。006 が実際に使った日本語版 Wikipedia コーパス(記事タイトル・リビジョン ID を
`src/data/wikipedia_manifests/ja_006_pretraining.json` に固定、約 24.58 MB)から、
006 と完全に同一の手順(訓練・検証に分割する前のコーパス全体から出現文字を語彙化する)
で再構築した。

## 構成

- 方式: 文字レベル(character-level)
- 語彙サイズ: {ja_tokenizer.vocab_size}(コーパスに出現したユニーク文字数、006 の本番
  実行結果と一致することを確認済み)

`tokenizer.json` は文字 → ID の対応表(`char_to_id`)のみを含む
(`src/data/text.py` の `load_character_level_tokenizer_json()` で読み込める)。
"""

code_model_card = f"""---
language: en
license: cc-by-nc-4.0
tags:
- ai-theories
- tokenizer
- byte-level-bpe
- provisional
---

# ai-theories 標準トークナイザ(コード・バイトレベル BPE、暫定)

`ai-theories`(https://github.com/kojikojiprg/ai-theories)プロジェクトの成果物。
バイトレベル Byte Pair Encoding(BPE、語彙サイズ {code_tokenizer.vocab_size})の
スクラッチ実装によるトークナイザ。

研究・教育目的で構築したものであり、品質保証は行っていない。商用・実運用での利用は
想定しない。

## 暫定(provisional)扱いについて

**このアーティファクトは暫定扱いである。**
[006. 小型 GPT の事前学習](https://github.com/kojikojiprg/ai-theories/blob/main/theories/02_pretraining/006_pretraining_small_gpt.ipynb)
はコードドメインで言語モデルの学習・トークナイザ条件の比較を行っておらず、コードドメイン
における「正解」となる設定(方式・語彙サイズ)はまだ確立していない。将来、コードドメインで
のトークナイザ比較実験が行われ、より適した設定が判明した場合はこのアーティファクトを
差し替える可能性がある。

現時点では、006 の本番実験で英語について選定された設定(バイトレベル BPE、語彙サイズ
8192、`max_chunk_bytes=64`)を仮採用している。

## 由来

[005. トークナイザ](https://github.com/kojikojiprg/ai-theories/blob/main/theories/02_pretraining/005_tokenizer.ipynb)
で導入された `load_code_corpus()`(`src/data/text.py`)により取得したコーパス
(本リポジトリ自身の `src/` 配下の Python ソースコードを連結したもの)から学習した。

**このコーパスはリポジトリ自身の内容に依存する。** `load_code_corpus()` はリポジトリの
`src/` を実行時点の内容で連結するため、取得結果は取得時点のリポジトリ内容に依存し、
将来のコミットでは変化しうる。取得に使ったコミットハッシュ: `{code_commit_hash}`
{_dirty_note}

## 構成

- 方式: バイトレベル BPE(byte-level Byte Pair Encoding)
- 語彙サイズ: {code_tokenizer.vocab_size}
- 事前分割チャンクの最大バイト数(`max_chunk_bytes`): 64
- 学習コーパスサイズ: {len(code_corpus.encode("utf-8")):,} バイト
"""

(en_artifact_dir / "README.md").write_text(en_model_card, encoding="utf-8")
(ja_artifact_dir / "README.md").write_text(ja_model_card, encoding="utf-8")
(code_artifact_dir / "README.md").write_text(code_model_card, encoding="utf-8")
print("3 件のモデルカードを .cache に書き出した(アップロードはまだ行っていない)")

3 件のモデルカードを .cache に書き出した(アップロードはまだ行っていない)


## アップロード(Google Colab で `DRY_RUN=False` として実行する)

`DRY_RUN=True` の間はアップロードを一切呼び出さない。`upload_tokenizer_artifact_to_hub()`
(`src/data/tokenizer.py`)は `DRY_RUN=False and IN_COLAB` の場合のみ、
`google.colab.userdata.get("HF_TOKEN")` で取得したトークンを使って呼び出す(008 と同じ
認証パターン)。Claude Code はこのセルにおいて、トークンの入力・環境変数への設定・実際の
アップロード実行を一切行わない。

In [7]:
ARTIFACTS = {
    EN_TARGET_REPO_ID: en_artifact_dir,
    JA_TARGET_REPO_ID: ja_artifact_dir,
    CODE_TARGET_REPO_ID: code_artifact_dir,
}

if not DRY_RUN and IN_COLAB:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
    for repo_id, artifact_dir in ARTIFACTS.items():
        print(f"アップロード中: {repo_id}")
        upload_tokenizer_artifact_to_hub(
            repo_id=repo_id,
            tokenizer_json_path=artifact_dir / "tokenizer.json",
            model_card_text=(artifact_dir / "README.md").read_text(encoding="utf-8"),
            token=token,
        )
        print(f"アップロード完了: https://huggingface.co/{repo_id}")
else:
    print(
        "アップロードをスキップした(DRY_RUN=True またはローカル実行のため)。"
        "本番実行は Google Colab で DRY_RUN=False の状態で行うこと。"
    )

アップロード中: kojikojiprg/ai-theories-tokenizer-en
アップロード完了: https://huggingface.co/kojikojiprg/ai-theories-tokenizer-en
アップロード中: kojikojiprg/ai-theories-tokenizer-ja
アップロード完了: https://huggingface.co/kojikojiprg/ai-theories-tokenizer-ja
アップロード中: kojikojiprg/ai-theories-tokenizer-code
アップロード完了: https://huggingface.co/kojikojiprg/ai-theories-tokenizer-code


## まとめ

- 英語(en): 008 のモデルリポジトリから取得した `tokenizer.json` をそのまま再利用した
  (語彙サイズ 8192 を確認、ラウンドトリップ検証済み)。009 との内容比較は、009 が
  Colab で本番実行中のためスキップした。
- 日本語(ja): 006 が実際に使ったコーパス取得手順(`load_wikipedia_corpus("ja", ...)`、
  訓練・検証分割前のコーパス全体から文字レベル語彙を構築)を正確に再現し、語彙サイズが
  006 の本番実行結果(4654)と一致することを確認した。ラウンドトリップ検証済み。
- コード(code): `load_code_corpus()` から取得したコーパスで英語と同一設定の
  バイトレベル BPE を学習した(暫定扱い)。ラウンドトリップ検証済み。コーパスの
  取得元コミットハッシュを記録した。
- 3 つのモデルカードを作成した(`cc-by-nc-4.0`、暫定扱いの明記、コミットハッシュの記載を含む)。
- **アップロードは実行していない。** こうじさんが Google Colab で `DRY_RUN=False`
  に切り替えて実行する必要がある。
- 009(`theories/02_pretraining/009_scaling_laws.ipynb`)のトークナイザ取得セルの更新は、
  009 の Colab 本番実行(スケーリング則の学習グリッド)が完了してから別途行う
  (本ノートブックでは行っていない)。